In [ ]:
import json

with open("../data/math/mathhard_processed_7B.jsonl", "r") as f:
    data = [json.loads(line) for line in f]
print(f"Total problems: {len(data)}")

Total problems: 1257


In [9]:
data[0]

{'id': 'mathhard_group_0',
 'question': 'Does there exist a complete norm on the vector space of all sequences in \\(\\mathbb{R}\\)?',
 'final_answer': 'Yes',
 'experiences': [{'question': 'Determine the integers $k$ for which there exists a smooth map $f : M \\to T^n$ of degree $k$, given that $M$ is a compact oriented smooth $n$-manifold with $H_{dR}^1(M)=0$.',
   'response': '\n\nTo determine the integers $ k $ for which there exists a smooth map $ f : M \\to T^n $ of degree $ k $, where $ M $ is a compact oriented smooth $ n $-manifold with $ H^1_{\\text{dR}}(M) = 0 $, we proceed by analyzing the topological and cohomological constraints imposed by the condition $ H^1_{\\text{dR}}(M) = 0 $ and the nature of the target space $ T^n $.\n\n---\n\n### Step 1: Understanding the Degree of a Map\n\nThe **degree** of a smooth map $ f : M \\to T^n $ is a topological invariant defined when both $ M $ and $ T^n $ are compact, oriented, and of the same dimension. It is an integer that measures 

In [ ]:
training_set = data[:1000]
validation_set = data[1000:]


# restruct the training set and convert to parquet files
train_res = []
for idx, item in enumerate(validation_set):
    instance_id = idx
    prompt = "I will provide you with sequential past experiences of attempts to solve a math problem. Please analyze each experience (e.g., whether the solution is correct, topic, etc.) and decide what memory operations to perform to store this information effectively. Use memory_insert, memory_update, or memory_delete operations as needed."
    chunks = []
    for exp in item['experiences']:
        chunk_item = ""
        chunk_item += f"[Question] {exp['question']}\n"
        chunk_item += f"[Attempted Solution] {exp['response']}\n"
        chunk_item += f"[Gold Answer] {exp['final_answer']}\n"
        chunk_item += f"[Topic] {exp['topic']}\n"
        chunks.append(chunk_item)
    questions_and_answers = [{
        "question": exp['question'],
        "answer": exp['final_answer']
    }]
    data_source = "math"
    metadata = {
        "topic": item['topic'],
        "data_source": data_source,
        "difficulty": item['difficulty']
    }
    num_chunks = len(chunks)
    num_questions = 1

    train_res.append({
        "instance_id": int(instance_id),
        "prompt": str(prompt),
        "chunks": json.dumps(chunks),
        "questions_and_answers": json.dumps(questions_and_answers),
        "data_source": str(data_source),
        "metadata": json.dumps(metadata),
        "num_chunks": int(num_chunks),
        "num_questions": int(num_questions)
    })

# store train_res into parquet file
import pandas as pd
train_df = pd.DataFrame(train_res)
train_df.to_parquet("../data/math/test.parquet", index=False)   